# Cell 1

In [ ]:
# =============================================================================
# Thesis — GeoAI Café Site Selection · Milan
# Phase 2 · Notebook 1 · Environment Setup and Data Ingestion (Bronze Layer)
# =============================================================================
#
# Purpose:
#   Install all required libraries, connect to Google Drive for persistent
#   storage, define the sample study area boundary, and pull all raw data
#   from every source. Nothing is transformed in this notebook. Data lands
#   exactly as received into the Bronze folder.
#
# Inputs:
#   - OpenStreetMap via Overpass API (osmnx)
#   - Urban Atlas land use polygons (static file, manual upload)
#   - Dati Lombardia population data (static file, manual upload)
#   - VIIRS night lights raster (static file, manual upload)
#
# Outputs (all saved to /Bronze on Google Drive):
#   - raw_cafes.geojson
#   - raw_transit_stops.geojson
#   - raw_road_network_nodes.geojson
#   - raw_road_network_edges.geojson
#   - raw_pois.geojson
#   - raw_urban_atlas.geojson
#   - raw_population.geojson
#   - raw_viirs.tif
#   - ingestion_log.txt
#
# CRS at ingestion: WGS84 EPSG:4326
# Target CRS for analysis: EPSG:32632 UTM Zone 32N (applied in Notebook 2)
# =============================================================================

# Cell 2

In [ ]:
# Install all required libraries
!pip install osmnx -q
!pip install h3 -q
!pip install geopandas -q
!pip install shapely -q
!pip install rasterio -q
!pip install pyproj -q
!pip install folium -q          # lightweight browser map for quick visual checks
!pip install mapclassify -q     # required by geopandas explore()

print("All libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.0 MB/s eta 0:00:00
All libraries installed successfully


# Cell 3

In [ ]:
import os
import datetime
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module='osmnx')

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
import folium

from shapely.geometry import Point

# Print versions for reproducibility documentation
print(f"osmnx:      {ox.__version__}")
print(f"geopandas:  {gpd.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"rasterio:   {rasterio.__version__}")

ox.settings.log_console = False
ox.settings.use_cache   = True
ox.settings.overpass_settings = '[out:json][timeout:180]'

INGESTION_DATE = None  # set at log-write time in Cell 12

print("\nGlobal settings configured:")
print(f"  ox.settings.use_cache         = {ox.settings.use_cache}")
print(f"  ox.settings.overpass_settings = {ox.settings.overpass_settings}")
print(f"  INGESTION_DATE will be set in Cell 12 at log-write time")

osmnx:      2.1.0
geopandas:  1.1.3
pandas:     2.2.2
numpy:      2.0.2
rasterio:   1.5.0

Global settings configured:
  ox.settings.use_cache         = True
  ox.settings.overpass_settings = [out:json][timeout:180]
  INGESTION_DATE will be set in Cell 12 at log-write time


# Cell 4

In [ ]:
# Connect Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Thesis'

BRONZE_PATH = os.path.join(PROJECT_ROOT, 'Bronze')
SILVER_PATH = os.path.join(PROJECT_ROOT, 'Silver')
GOLD_PATH   = os.path.join(PROJECT_ROOT, 'Gold')

# Create all directories if they do not exist yet
for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    os.makedirs(path, exist_ok=True)
    print(f"Ready: {path}")

print("\nDrive connected and folder structure verified")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready: /content/drive/MyDrive/Thesis/Bronze
Ready: /content/drive/MyDrive/Thesis/Silver
Ready: /content/drive/MyDrive/Thesis/Gold

Drive connected and folder structure verified


# Cell 5

In [ ]:
# Define the study area for Phase 2 (full Milan — all 9 Municipios)
#
# TWO boundaries are defined here:
#
#   study_area         — the exact municipio boundaries
#                        used for H3 grid generation and final clipping
#
#   study_area_buffered — the municipio boundaries expanded by BUFFER_M metres
#                         used for ALL OSM data ingestion queries
#                         ensures cells near the boundary have complete
#                         spatial context — transit hubs, POIs, and road
#                         network just outside the boundary are included
#                         in feature calculations
#
# Without the buffer, a cell 50 metres from the boundary edge would appear
# to have no nearby metro station if that station sits just outside the
# municipio border — a false signal that corrupts feature engineering.

SAMPLE_NEIGHBOURHOODS = [
    "Municipio 1, Milan, Italy",
    "Municipio 2, Milan, Italy",
    "Municipio 3, Milan, Italy",
    "Municipio 4, Milan, Italy",
    "Municipio 5, Milan, Italy",
    "Municipio 6, Milan, Italy",
    "Municipio 7, Milan, Italy",
    "Municipio 8, Milan, Italy",
    "Municipio 9, Milan, Italy"
]

BUFFER_M = 1500

print("Retrieving study area boundaries from OSM (Phase 2 — full Milan)...")

boundary_parts = []
for neighbourhood in SAMPLE_NEIGHBOURHOODS:
    try:
        boundary = ox.geocode_to_gdf(neighbourhood)
        boundary_parts.append(boundary)
        print(f"  Retrieved: {neighbourhood}")
    except Exception as e:
        print(f"  Failed: {neighbourhood} — {e}")

assert len(boundary_parts) == len(SAMPLE_NEIGHBOURHOODS), (
    f"Only {len(boundary_parts)}/{len(SAMPLE_NEIGHBOURHOODS)} boundaries "
    f"retrieved — check OSM geocoding for failed entries above."
)

# Merge all boundaries into a single study area polygon
study_area           = pd.concat(boundary_parts, ignore_index=True)
study_area_dissolved = study_area.dissolve()

# Reproject to EPSG:32632 to apply buffer in metres
study_area_32632 = study_area_dissolved.to_crs('EPSG:32632')

# Apply buffer — this is the ingestion boundary for all OSM queries
study_area_buffered_32632 = study_area_32632.copy()
study_area_buffered_32632['geometry'] = (
    study_area_32632.geometry.buffer(BUFFER_M)
)

# Reproject buffered boundary back to WGS84 for OSM queries
study_area_buffered = study_area_buffered_32632.to_crs('EPSG:4326')

# Extract polygons for use in subsequent cells
study_polygon_wgs84          = study_area_dissolved.geometry.iloc[0]
study_polygon_buffered_wgs84 = study_area_buffered.geometry.iloc[0]

# Sanity check
bounds          = study_area_dissolved.total_bounds
bounds_buffered = study_area_buffered.total_bounds

print(f"\nStudy area bounding box (WGS84 — exact boundary):")
print(f"  West: {bounds[0]:.4f}  South: {bounds[1]:.4f}")
print(f"  East: {bounds[2]:.4f}  North: {bounds[3]:.4f}")

print(f"\nBuffered ingestion area (WGS84 — {BUFFER_M}m buffer):")
print(f"  West: {bounds_buffered[0]:.4f}  South: {bounds_buffered[1]:.4f}")
print(f"  East: {bounds_buffered[2]:.4f}  North: {bounds_buffered[3]:.4f}")

assert study_area_dissolved.crs is not None and study_area_dissolved.crs.to_epsg() == 4326, (
    "Study area CRS mismatch after dissolve — expected EPSG:4326, "
    f"got: {study_area_dissolved.crs}"
)

study_area_dissolved.to_file(
    os.path.join(BRONZE_PATH, 'raw_study_area_boundary.geojson'),
    driver='GeoJSON'
)
study_area_buffered.to_file(
    os.path.join(BRONZE_PATH, 'raw_study_area_buffered.geojson'),
    driver='GeoJSON'
)

print(f"\nBoth boundaries saved to Bronze")
print(f"  raw_study_area_boundary.geojson  — used for H3 grid generation")
print(f"  raw_study_area_buffered.geojson  — used for all OSM ingestion")

Retrieving study area boundaries from OSM (Phase 2 — full Milan)...
  Retrieved: Municipio 1, Milan, Italy
  Retrieved: Municipio 2, Milan, Italy
  Retrieved: Municipio 3, Milan, Italy
  Retrieved: Municipio 4, Milan, Italy
  Retrieved: Municipio 5, Milan, Italy
  Retrieved: Municipio 6, Milan, Italy
  Retrieved: Municipio 7, Milan, Italy
  Retrieved: Municipio 8, Milan, Italy
  Retrieved: Municipio 9, Milan, Italy

Study area bounding box (WGS84 — exact boundary):
  West: 9.0409  South: 45.3867
  East: 9.2781  North: 45.5358

Buffered ingestion area (WGS84 — 1500m buffer):
  West: 9.0217  South: 45.3732
  East: 9.2973  North: 45.5493

Both boundaries saved to Bronze
  raw_study_area_boundary.geojson  — used for H3 grid generation
  raw_study_area_buffered.geojson  — used for all OSM ingestion


# Cell 6

In [ ]:
print("Pulling café locations from OSM...")

cafes = ox.features_from_polygon(
    study_polygon_buffered_wgs84,
    tags={'amenity': 'cafe'}
)

assert len(cafes) > 0, (
    "OSM café query returned 0 records — check Overpass API connectivity "
    "and query tags {'amenity': 'cafe'}."
)

cafes = cafes.reset_index()

cols_to_keep = ['geometry', 'osmid', 'name', 'amenity', 'opening_hours']
cols_present = [c for c in cols_to_keep if c in cafes.columns]
cafes = cafes[cols_present].copy()

print(f"  Retrieved {len(cafes)} café records")
print(f"  Geometry types: {cafes.geometry.geom_type.value_counts().to_dict()}")
print(f"  CRS: {cafes.crs}")

cafe_path = os.path.join(BRONZE_PATH, 'raw_cafes.geojson')
cafes.to_file(cafe_path, driver='GeoJSON')
print(f"  Saved to Bronze: raw_cafes.geojson")

Pulling café locations from OSM...
  Retrieved 2014 café records
  Geometry types: {'Point': 1975, 'Polygon': 39}
  CRS: epsg:4326
  Saved to Bronze: raw_cafes.geojson


# Cell 7

In [ ]:
print("Pulling transit stops from OSM...")

# Metro stations
metro = ox.features_from_polygon(
    study_polygon_buffered_wgs84,
    tags={'station': 'subway'}
)

# Bus/tram stop positions (rail/tram rolling-stock positions in OSM)
bus_tram = ox.features_from_polygon(
    study_polygon_buffered_wgs84,
    tags={'public_transport': 'stop_position'}
)

# Bus stops — standard OSM tag for roadside bus stop signs/shelters
bus_stop = ox.features_from_polygon(
    study_polygon_buffered_wgs84,
    tags={'highway': 'bus_stop'}
)

# Platforms — passenger waiting areas for all public transport modes
platform = ox.features_from_polygon(
    study_polygon_buffered_wgs84,
    tags={'public_transport': 'platform'}
)

assert len(metro) > 0, (
    "OSM metro query returned 0 records — check Overpass API connectivity "
    "and query tags {'station': 'subway'}."
)
assert len(bus_tram) > 0, (
    "OSM bus/tram stop_position query returned 0 records — check Overpass "
    "API connectivity and query tags {'public_transport': 'stop_position'}."
)
assert len(bus_stop) > 0, (
    "OSM bus_stop query returned 0 records — check Overpass API connectivity "
    "and query tags {'highway': 'bus_stop'}."
)
assert len(platform) > 0, (
    "OSM platform query returned 0 records — check Overpass API connectivity "
    "and query tags {'public_transport': 'platform'}."
)

metro['transit_type']    = 'metro'
bus_tram['transit_type'] = 'bus_tram'
bus_stop['transit_type'] = 'bus_stop'
platform['transit_type'] = 'platform'

# Combine and reset index
# NOTE: pd.concat with ignore_index=True drops the OSM MultiIndex but does NOT
# deduplicate. A physical stop is often present under multiple OSM tags
# (e.g. public_transport=stop_position AND highway=bus_stop), inflating raw
# counts. Spatial deduplication (5 m STRtree + union-find) is deferred to
# Notebook 2 Cell 8 where it is applied alongside café deduplication.
# The raw count below therefore includes duplicates — the Silver layer count
# is the authoritative figure after dedup.
transit = pd.concat([metro, bus_tram, bus_stop, platform], ignore_index=True)
transit = transit.reset_index(drop=True)

# Keep geometry and transit type
transit = transit[['geometry', 'transit_type']].copy()

print(f"  Metro stations:                  {len(metro)}")
print(f"  Bus/tram stop positions:         {len(bus_tram)}")
print(f"  Bus stops (highway=bus_stop):    {len(bus_stop)}")
print(f"  Platforms:                       {len(platform)}")
print(f"  Total transit records (raw, pre-dedup): {len(transit)}")
print(f"  NOTE: transit deduplication (5 m threshold) applied in Notebook 2 Cell 8.")
print(f"  Raw count above inflates transit_density if used directly — use Silver layer.")

# Save to Bronze
transit_path = os.path.join(BRONZE_PATH, 'raw_transit_stops.geojson')
transit.to_file(transit_path, driver='GeoJSON')
print(f"  Saved to Bronze: raw_transit_stops.geojson")

Pulling transit stops from OSM...
  Metro stations:                  124
  Bus/tram stop positions:         4370
  Bus stops (highway=bus_stop):    3621
  Platforms:                       7021
  Total transit records (raw, pre-dedup): 15136
  NOTE: transit deduplication (5 m threshold) applied in Notebook 2 Cell 8.
  Raw count above inflates transit_density if used directly — use Silver layer.
  Saved to Bronze: raw_transit_stops.geojson


# Cell 8

In [ ]:
print("Pulling road network from OSM...")
print("This may take several minutes for the full study area...")

# NOTE: ox.settings.overpass_settings = '[out:json][timeout:180]' is set in
# Cell 3 (global settings block) and applies here without re-assignment.

network = ox.graph_from_polygon(
    study_polygon_buffered_wgs84,
    network_type='walk'
)

nodes, edges = ox.graph_to_gdfs(network)

print(f"  Network nodes: {len(nodes)}")
print(f"  Network edges: {len(edges)}")
print(f"  CRS: {nodes.crs}")

assert len(nodes) > 50_000, (
    f"Road network nodes suspiciously low: {len(nodes)} — "
    f"possible Overpass API truncation. Re-run after checking network access, "
    f"or increase timeout via ox.settings.overpass_settings."
)

nodes = nodes.reset_index()   # osmid becomes a regular column

edges = edges.reset_index()   # u, v, key become regular columns

# Save both to Bronze
nodes_path = os.path.join(BRONZE_PATH, 'raw_road_network_nodes.geojson')
edges_path = os.path.join(BRONZE_PATH, 'raw_road_network_edges.geojson')

nodes.to_file(nodes_path, driver='GeoJSON')
edges.to_file(edges_path, driver='GeoJSON')

print(f"  Saved to Bronze: raw_road_network_nodes.geojson")
print(f"  Saved to Bronze: raw_road_network_edges.geojson")

Pulling road network from OSM...
This may take several minutes for the full study area...
  Network nodes: 114046
  Network edges: 337620
  CRS: epsg:4326
  Saved to Bronze: raw_road_network_nodes.geojson
  Saved to Bronze: raw_road_network_edges.geojson


# Cell 9

In [ ]:
print("Pulling general POIs from OSM...")

poi_tags = {
    'shop':     True,           # all retail shops
    'amenity':  True,           # all amenity types
    'tourism':  True,           # tourist attractions
    'building': ['office'],     # office buildings
    'landuse':  ['commercial',  # commercial zones
                 'retail'],
    'office':   True            # office nodes
}

pois = ox.features_from_polygon(
    study_polygon_buffered_wgs84,
    tags=poi_tags
)

assert len(pois) > 0, (
    "OSM POI query returned 0 records — check Overpass API connectivity "
    f"and query tags: {list(poi_tags.keys())}."
)

pois = pois.reset_index()

category_cols = ['geometry', 'osmid', 'shop', 'amenity',
                 'tourism', 'building', 'landuse', 'office', 'name']
cols_present = [c for c in category_cols if c in pois.columns]
pois = pois[cols_present].copy()

print(f"  Retrieved {len(pois)} POI records")
print(f"  CRS: {pois.crs}")

# Save to Bronze
pois_path = os.path.join(BRONZE_PATH, 'raw_pois.geojson')
pois.to_file(pois_path, driver='GeoJSON')
print(f"  Saved to Bronze: raw_pois.geojson")

Pulling general POIs from OSM...
  Retrieved 60271 POI records
  CRS: epsg:4326
  Saved to Bronze: raw_pois.geojson


# Cell 10

In [ ]:
# =============================================================================
# Cell 10 — Verify Static Files (Urban Atlas 2021 data and Population)
# =============================================================================
#
# Bronze layer rule: data lands exactly as received. No transformations.
# This cell only verifies files are present. All cleaning, reprojection,
# parsing, and density calculations happen in Notebook 2 (Silver layer).
#
# Files expected in Bronze folder:
#   raw_urban_atlas.fgb   — downloaded from Copernicus Land Service
#   raw_population.csv    — downloaded from Dati Lombardia
# =============================================================================

# =============================================================================
# Where to download the files
# =============================================================================
#
# Urban Atlas:
#   Go to: https://land.copernicus.eu/local/urban-atlas
#   Download Urban Atlas Land Cover/Land Use 2024 (vector) Pre-packaged
#   for the Milano FUA. File arrives as .fgb (FlatGeobuf) via email link.
#   Upload the .fgb file to your Bronze folder on Drive.
#   Comment: The column names code_2021 and class_2021 refer to the reference
#   year of the land use survey — meaning the land use classification was
#   conducted based on satellite imagery and field data from 2021. The file
#   date inside the file we downloaded is simply when Copernicus published and
#   packaged that dataset. The underlying land use data itself is from the 2021
#   survey cycle.
#
# Dati Lombardia Population:
#   Go to: https://www.dati.lombardia.it
#   Download: CITTA METROPOLITANA MILANO - 2019 Popolazione residente
#   popolazione e superficie legale al 31.12
#   Download as CSV and upload to Bronze folder as raw_population.csv


URBAN_ATLAS_FILENAME = 'raw_urban_atlas.fgb'
POPULATION_FILENAME  = 'raw_population.csv'
ISTAT_FILENAME       = 'R03_21_WGS84.shp'   # canonical population source

urban_atlas_path = os.path.join(BRONZE_PATH, URBAN_ATLAS_FILENAME)
population_path  = os.path.join(BRONZE_PATH, POPULATION_FILENAME)
istat_path       = os.path.join(BRONZE_PATH, ISTAT_FILENAME)

# Determine which population source is available
istat_available = os.path.exists(istat_path)

# Urban Atlas — file presence check only (Bronze layer rule: no reads)
# Actual read with OGR_GEOJSON_MAX_OBJ_SIZE=0 is performed in NB2 Cell 4
if os.path.exists(urban_atlas_path):
    print(f"Urban Atlas: raw_urban_atlas.fgb — present")
    print(f"  (File will be read in Notebook 2 Silver processing)")
else:
    print("MISSING: raw_urban_atlas.fgb")
    print("  Please download Urban Atlas 2024 Pre-packaged for Milano FUA")
    print("  from: https://land.copernicus.eu/local/urban-atlas")
    print(f"  Upload the .fgb file to: {BRONZE_PATH}")

urban_atlas = None
print()

# ISTAT 2021 census tracts — primary population source
# Presence check only — file is NOT opened here (Bronze layer rule).
# Sidecar files (.dbf, .prj, .shx) are required but not validated here;
# NB2 Cell 9 will raise a descriptive error if they are missing.
if istat_available:
    print(f"Population (primary): ISTAT 2021 R03_21_WGS84.shp — "
          f"present (not read — Bronze layer rule)")
else:
    print(f"Population (primary): ISTAT 2021 R03_21_WGS84.shp — MISSING")
    print(f"  Upload R03_21_WGS84.shp (+ .shx, .dbf, .prj) to: {BRONZE_PATH}")

# Population CSV — fallback source
# Pre-flight column check only — confirms readability before NB2 runs.
# No transforms applied; Bronze layer rule maintained.
if os.path.exists(population_path):
    population_csv = pd.read_csv(population_path, encoding='utf-8')
    print(f"Population CSV (fallback): present (pre-flight column check only — "
          f"no transforms applied)")
    print(f"  Rows:    {len(population_csv)}")
    print(f"  Columns: {list(population_csv.columns)}")
    print(f"\n  First 3 rows (raw — no transformations applied):")
    print(population_csv.head(3).to_string())
else:
    print("Population CSV (fallback): raw_population.csv — MISSING")
    print("  Please download population data from:")
    print("  https://www.dati.lombardia.it")
    print("  Search: CITTA METROPOLITANA MILANO 2019 Popolazione residente")
    print(f"  Download as CSV and upload to: {BRONZE_PATH}")
    print(f"  Rename the file to: raw_population.csv")
    population_csv = None

Urban Atlas: raw_urban_atlas.fgb — present
  (File will be read in Notebook 2 Silver processing)

Population (primary): ISTAT 2021 R03_21_WGS84.shp — present (not read — Bronze layer rule)
Population CSV (fallback): present (pre-flight column check only — no transforms applied)
  Rows:    133
  Columns: ['Comune', 'Zona_Omogenea', 'Codice_catastale', 'Codice_Istat', 'Popolazione_residente__al_31-12-2019', 'Popolazione_legale_2011', 'Superficie_Kmq']

  First 3 rows (raw — no transformations applied):
          Comune         Zona_Omogenea Codice_catastale  Codice_Istat Popolazione_residente__al_31-12-2019 Popolazione_legale_2011  Superficie_Kmq
0  Abbiategrasso  Magentino Abbiatense             A010         15002                               32,855                  30,994           47.78
1      Albairate  Magentino Abbiatense             A127         15005                                4,735                   4,621           14.98
2       Arconate         Alto Milanese             A3

# Cell 11

In [ ]:
# How to obtain:
#   Go to: https://eogdata.mines.edu/products/vnl/
#   Download the annual VNL V2 composite for the most recent year
#   covering Italy (tile 75N060E or use the global file clipped to Europe)
#   Upload the .tif file to your Bronze folder on Drive

VIIRS_FILENAME = 'raw_viirs.tif'
viirs_path = os.path.join(BRONZE_PATH, VIIRS_FILENAME)

if os.path.exists(viirs_path):
    with rasterio.open(viirs_path) as src:
        print(f"VIIRS raster loaded:")
        print(f"  CRS:        {src.crs}")
        print(f"  Resolution: {src.res}")
        print(f"  Bounds:     {src.bounds}")
        print(f"  Bands:      {src.count}")
else:
    print("MISSING: raw_viirs.tif")
    print("  Please download VIIRS VNL V2 annual composite from:")
    print("  https://eogdata.mines.edu/products/vnl/")
    print(f"  Upload the .tif file to: {BRONZE_PATH}")

VIIRS raster loaded:
  CRS:        EPSG:4326
  Resolution: (0.0041666667, 0.0041666667)
  Bounds:     BoundingBox(left=-180.00208333335, bottom=-65.00208445335001, right=180.00208621335, top=75.00208333335)
  Bands:      1


# Cell 12

In [ ]:
# Write a log file recording exactly what was pulled, when, and from where
# This is your reproducibility record — cite this date in the thesis
# when describing the temporal currency of your data

INGESTION_DATE = datetime.datetime.now().strftime("%Y-%m-%d")
print(f"Ingestion date (recorded at log-write time): {INGESTION_DATE}")

istat_status_str = (
    'present (not read — Bronze layer rule)'
    if istat_available
    else 'MISSING'
)

# Population CSV status: "present (pre-flight column check only)" or "MISSING"
pop_csv_status_str = (
    'present (pre-flight column check only)'
    if os.path.exists(population_path)
    else 'MISSING'
)

log_lines = [
    f"Thesis Phase 2 — Data Ingestion Log",
    f"====================================",
    f"Ingestion date:          {INGESTION_DATE}",
    f"Study area:              {', '.join(SAMPLE_NEIGHBOURHOODS)}",
    f"Ingestion buffer:        {BUFFER_M} metres beyond study boundary",
    f"",
    f"OSM Data (pulled via osmnx from Overpass API)",
    f"  Cafes:                 {len(cafes)} records",
    f"  Transit stops:         {len(transit)} records",
    f"  Road network:          {len(nodes)} nodes, {len(edges)} edges",
    f"  General POIs:          {len(pois)} records",
    f"",
    f"Static Files",
    f"  Urban Atlas 2021:      {'present' if os.path.exists(urban_atlas_path) else 'MISSING'}",
    f"  Population (primary)   ISTAT 2021 R03_21_WGS84.shp: {istat_status_str}",
    f"  Population (fallback)  Dati Lombardia 2019 CSV: {pop_csv_status_str}",
    f"  VIIRS Night Lights 2024: {'present' if os.path.exists(viirs_path) else 'MISSING'}",
    f"",
    f"CRS at ingestion:        WGS84 EPSG:4326",
    f"Target CRS:              EPSG:32632 UTM Zone 32N (applied in Notebook 2)",
    f"",
    f"Notes:",
    f"  OSM data ingested within {BUFFER_M}m buffer beyond study boundary.",
    f"  Buffer ensures complete spatial context for cells near borders.",
    f"  H3 grid generation uses exact boundary only (no buffer).",
    f"  All data saved exactly as received to Bronze layer.",
    f"  Deduplication and reprojection applied in Notebook 2.",
    f"  ISTAT .shp presence checked via os.path.exists() only — file not opened.",
    f"  Sidecar file (.dbf/.prj/.shx) validation deferred to NB2 Cell 9.",
]

log_path = os.path.join(BRONZE_PATH, 'ingestion_log.txt')
with open(log_path, 'w') as f:
    f.write('\n'.join(log_lines))

print('\n'.join(log_lines))
print(f"\nLog saved to: {log_path}")

Ingestion date (recorded at log-write time): 2026-04-26
Thesis Phase 2 — Data Ingestion Log
Ingestion date:          2026-04-26
Study area:              Municipio 1, Milan, Italy, Municipio 2, Milan, Italy, Municipio 3, Milan, Italy, Municipio 4, Milan, Italy, Municipio 5, Milan, Italy, Municipio 6, Milan, Italy, Municipio 7, Milan, Italy, Municipio 8, Milan, Italy, Municipio 9, Milan, Italy
Ingestion buffer:        1500 metres beyond study boundary

OSM Data (pulled via osmnx from Overpass API)
  Cafes:                 2014 records
  Transit stops:         15136 records
  Road network:          114046 nodes, 337620 edges
  General POIs:          60271 records

Static Files
  Urban Atlas 2021:      present
  Population (primary)   ISTAT 2021 R03_21_WGS84.shp: present (not read — Bronze layer rule)
  Population (fallback)  Dati Lombardia 2019 CSV: present (pre-flight column check only)
  VIIRS Night Lights 2024: present

CRS at ingestion:        WGS84 EPSG:4326
Target CRS:              

# Cell 13

In [ ]:
# =============================================================================
# Cell 13 — Quick visual check that the data landed correctly
# =============================================================================

m = folium.Map(location=[45.4642, 9.1900], zoom_start=12, tiles='CartoDB positron')

folium.GeoJson(
    study_area_dissolved.__geo_interface__,
    name='Study area',
    style_function=lambda x: {'fillColor': 'none', 'color': '#333333', 'weight': 2}
).add_to(m)

cafe_points = cafes[cafes.geometry.geom_type == 'Point'].dropna(subset=['geometry'])
cafe_sample_filtered = cafe_points.head(500)

# FIX (Batch 8 — NB1 Minor): Report how many polygon-typed café features are
# excluded from the map so the "Red dots = café locations" note is traceable.
n_polygon_cafes = len(cafes) - len(cafe_points)
if n_polygon_cafes > 0:
    print(f"  NOTE: {n_polygon_cafes} polygon café feature(s) excluded from map "
          f"(Way-derived buildings in OSM — centroids not plotted here).")

for _, row in cafe_sample_filtered.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=3,
            color='#E8593C',
            fill=True,
            fill_opacity=0.7,
            popup=str(row.get('name', 'Café'))
        ).add_to(m)

folium.LayerControl().add_to(m)

print("Showing study area boundary and first 500 café Point locations")
print("Red dots = café locations from OSM (Point geometries only)")
m

  NOTE: 39 polygon café feature(s) excluded from map (Way-derived buildings in OSM — centroids not plotted here).
Showing study area boundary and first 500 café Point locations
Red dots = café locations from OSM (Point geometries only)


# Cell 14

In [ ]:
# Final summary — verify all Bronze outputs are present before
# moving to Notebook 2

print("=" * 60)
print("NOTEBOOK 1 COMPLETE — Bronze Layer Ingestion Summary")
print("=" * 60)

# Population check: pipeline can proceed if EITHER the primary ISTAT shapefile
# OR the fallback CSV is present. Both missing is the only blocking condition.
istat_present      = os.path.exists(istat_path)
pop_csv_present    = os.path.exists(population_path)
pop_ok             = istat_present or pop_csv_present

if istat_present:
    pop_status = "OK"
    pop_label  = "R03_21_WGS84.shp (ISTAT 2021 — primary)"
elif pop_csv_present:
    pop_status = "OK"
    pop_label  = "raw_population.csv (Dati Lombardia 2019 — fallback)"
else:
    pop_status = "MISSING"
    pop_label  = "R03_21_WGS84.shp AND raw_population.csv — both missing"

# All non-population files that must be present
required_files = [
    ('raw_study_area_boundary.geojson',  ''),
    ('raw_study_area_buffered.geojson',  ''),
    ('raw_cafes.geojson',                ''),
    ('raw_transit_stops.geojson',        ''),
    ('raw_road_network_nodes.geojson',   ''),
    ('raw_road_network_edges.geojson',   ''),
    ('raw_pois.geojson',                 ''),
    ('raw_urban_atlas.fgb',              ''),
    ('raw_viirs.tif',                    ''),
    ('ingestion_log.txt',                ''),
]

all_present = True
for filename, annotation in required_files:
    filepath = os.path.join(BRONZE_PATH, filename)
    status = "OK" if os.path.exists(filepath) else "MISSING"
    if status == "MISSING":
        all_present = False
    print(f"  [{status}] {filename}{annotation}")

# Print population status separately (pass if either source present)
print(f"  [{pop_status}] {pop_label}")
if not pop_ok:
    all_present = False

print()
if all_present:
    print("All Bronze files present. Ready to proceed to Notebook 2.")
else:
    print("Some files are missing. Complete the manual uploads above")
    print("before proceeding to Notebook 2.")

NOTEBOOK 1 COMPLETE — Bronze Layer Ingestion Summary
  [OK] raw_study_area_boundary.geojson
  [OK] raw_study_area_buffered.geojson
  [OK] raw_cafes.geojson
  [OK] raw_transit_stops.geojson
  [OK] raw_road_network_nodes.geojson
  [OK] raw_road_network_edges.geojson
  [OK] raw_pois.geojson
  [OK] raw_urban_atlas.fgb
  [OK] raw_viirs.tif
  [OK] ingestion_log.txt
  [OK] R03_21_WGS84.shp (ISTAT 2021 — primary)

All Bronze files present. Ready to proceed to Notebook 2.
